In [1]:
from data import om_cn

In [2]:
print('历史数据入库')
om_cn.save_archive()

历史数据入库
.. 历史数据量：126144; 
.. 库内最新日期：2025-10-16; 
.. 请求日期区间：2025-10-13~2025-10-16; 
.. 实际写入区间：2025-10-13~2025-10-16; 
.. 写入成功


In [3]:
print('读取历史数据')
archive_df = om_cn.read_archive()
print(archive_df['date'].min(), '~', archive_df['date'].max())

读取历史数据
.. 历史数据量：126144; 
2015-01-01 ~ 2025-10-16


In [4]:
print('获取预测数据')
forecast_df = om_cn.get_forecast(archive_df['date'].max())

获取预测数据
.. 请求日期区间：2025-10-17~2025-10-31; 
.. 预测数据量：480; 


In [5]:
print('数据加工')
process_df = om_cn.data_process(archive_df, forecast_df)

数据加工
.. 处理后数据量：31632; 


In [6]:
print('绘制图像')
cities_df = om_cn.read_cities()
charts_df = om_cn.read_charts()
forecast_after = archive_df['date'].max()

# 制图
import src.plt_charts as charts
import matplotlib.pyplot as plt
for i in cities_df.index:
    # 城市参数
    country = cities_df.loc[i]['country']
    city = cities_df.loc[i]['city']
    tag = cities_df.loc[i]['tag']
    code = cities_df.loc[i]['code']
    df = process_df[ process_df['city_code'] == code ].copy()
    for j in charts_df.index:
        params = {
            'min_history_year': charts_df.loc[j]['min_history_year'],
            'forecast_after':forecast_after,
            'ylabel': charts_df.loc[j]['y_label'],
            'title': charts_df.loc[j]['title'] + city + ', ' + country
        }
        if charts_df.loc[j]['variable'] == 'degree_day':
            params['xlim'] = (105, 260)
        chart = charts.day_annul_plot(df, charts_df.loc[j]['variable'], **params)
        path = f'./charts/cn/{i:02d}_{code}_{j:02d}_{charts_df.loc[j]['variable']}.jpg'
        chart.savefig(path, dpi=300)
        plt.close()
print('.. 绘制完成')

绘制图像
.. 绘制完成


In [7]:
print('合成大图')
file_lists = []
for i in cities_df.index:
    code = cities_df.loc[i]['code']
    for j in charts_df.index:
        path = f'./charts/cn/{i:02d}_{code}_{j:02d}_{charts_df.loc[j]['variable']}.jpg'
        file_lists.append(path)
charts.merge2grid(file_lists, cities_df.shape[0], charts_df.shape[0], './charts/cn/merge_cn.jpg')
print('.. 合成完成')

合成大图
